# Chapter 9 — Water\u2013Hydrocarbon Systems: Scientific Figures

**Figures:**
1. Water solubility in methane (CPA vs SRK vs experimental)
2. Methane solubility in water at high pressures
3. Water content of natural gas vs temperature and pressure
4. LLE prediction: water\u2013n-alkane series (C5\u2013C10)

In [1]:
import importlib, subprocess, sys
try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False); ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
except Exception:
    try: import neqsim
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
print(f"NeqSim mode: {NEQSIM_MODE}")

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim\src\main\resources
  3. C:\Users\ESOL\.m2\repository\com\h2database\h2\2.4.240\h2-2.4.240.jar
  4. C:\Users\ESOL\.m2\repository\org\apache\logging\log4j\log4j-api\2.25.4\log4j-api-2.25.4.jar
  5. C:\Users\ESOL\.m2\repository\org\apache\logging\log4j\log4j-core\2.25.4\log4j-core-2.25.4.jar
  6. C:\Users\ESOL\.m2\repository\com\thoughtworks\xstream\xstream\1.4.21\xstream-1.4.21.jar
  7. C:\Users\ESOL\.m2\repository\io\github\x-stream\mxparser\1.2.2\mxparser-1.2.2.jar
  8. C:\Users\ESOL\.m2\repository\xmlpull\xmlpull\1.1.3.1\xmlpull-1.1.3.1.jar
  9. C:\Users\ESOL\.m2\repository\org\apache\commons\commons-lang3\3.20.0\commons-lang3-3.20.0.jar
  10. C:\Users\ESOL\.m2\repository\org\apache\commons\commons-math3\3.6.1\commons-math3-3.6.1.jar
  11. C:\Users\ESOL\.m2\repository\org\ejml\ejml-all\0.44.0\ejml-all-0.44.0.jar
  12


JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim mode: devtools


In [2]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = ["#2171b5", "#e6550d", "#31a354", "#756bb1", "#e7298a", "#66a61e"]
BLUE, ORANGE, GREEN, PURPLE, PINK, LIME = PALETTE
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 10, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "xtick.direction": "in", "ytick.direction": "in", "axes.linewidth": 0.6, "lines.linewidth": 1.2, "grid.linewidth": 0.3, "grid.alpha": 0.4, "savefig.dpi": 300, "figure.dpi": 150})
FIGURES_DIR = Path("../figures"); FIGURES_DIR.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")
R = 8.314

In [3]:
if NEQSIM_MODE == "pip":
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
else:
    from neqsim import jneqsim
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

## Figure 9.1 — Water Content of Methane Gas (CPA vs SRK)

In [4]:
# Water content of gas at P=100 bar over temperature range
temps_C = np.arange(5, 105, 5)
data_cpa = {"T": [], "y_w_ppm": []}
data_srk = {"T": [], "y_w_ppm": []}

for T_C in temps_C:
    for label, cls, mr, data in [("CPA", SystemSrkCPAstatoil, 10, data_cpa),
                                  ("SRK", SystemSrkEos, "classic", data_srk)]:
        try:
            f = cls(273.15 + float(T_C), 100.0)
            f.addComponent("methane", 0.95)
            f.addComponent("water", 0.05)
            if isinstance(mr, int): f.setMixingRule(mr)
            else: f.setMixingRule(mr)
            f.setMultiPhaseCheck(True)
            ops = ThermodynamicOperations(f)
            ops.TPflash()
            f.initProperties()
            # Water in gas phase
            y_w = float(f.getPhase("gas").getComponent("water").getx()) * 1e6  # ppm
            data["T"].append(float(T_C))
            data["y_w_ppm"].append(y_w)
        except Exception:
            pass

# Experimental data (Olds et al., 1942; Rigby & Prausnitz, 1968) — approximate
exp_T = [25, 40, 60, 80, 100]
exp_yw = [550, 1300, 4200, 12000, 35000]

fig, ax = plt.subplots(figsize=(3.5, 3.0))
ax.semilogy(data_cpa["T"], data_cpa["y_w_ppm"], color=BLUE, linewidth=1.5, label="CPA")
ax.semilogy(data_srk["T"], data_srk["y_w_ppm"], color=ORANGE, linewidth=1.5, linestyle="--", label="SRK")
ax.semilogy(exp_T, exp_yw, 'ko', markersize=5, label="Experimental")
ax.set_xlabel("Temperature (\u00b0C)")
ax.set_ylabel("Water in gas (ppm mole)")
ax.set_title("Water content of methane at 100 bar")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3, which="both")
fig.tight_layout()
save(fig, "fig_ch09_01_water_content_methane.png")

Saved: fig_ch09_01_water_content_methane.png


## Figure 9.2 — Methane Solubility in Water at High Pressure

In [5]:
# Methane solubility in water at 25°C vs pressure
pressures_bar = np.arange(10, 510, 20)
x_ch4_cpa, x_ch4_srk = [], []

for P in pressures_bar:
    for label, cls, mr, x_list in [("CPA", SystemSrkCPAstatoil, 10, x_ch4_cpa),
                                    ("SRK", SystemSrkEos, "classic", x_ch4_srk)]:
        try:
            f = cls(273.15 + 25.0, float(P))
            f.addComponent("methane", 0.99)
            f.addComponent("water", 0.01)
            if isinstance(mr, int): f.setMixingRule(mr)
            else: f.setMixingRule(mr)
            f.setMultiPhaseCheck(True)
            ops = ThermodynamicOperations(f)
            ops.TPflash()
            f.initProperties()
            # Methane in aqueous phase
            nph = int(f.getNumberOfPhases())
            for ph in range(nph):
                pt = str(f.getPhase(ph).getPhaseTypeName())
                if "aqueous" in pt.lower() or ("liquid" in pt.lower() and ph > 0):
                    x_m = float(f.getPhase(ph).getComponent("methane").getx()) * 1e4  # x10^4
                    x_list.append(x_m)
                    break
            else:
                x_list.append(np.nan)
        except Exception:
            x_list.append(np.nan)

# Experimental data (Culberson & McKetta, 1951)
exp_P = [50, 100, 200, 300, 500]
exp_x = [1.5, 2.5, 3.5, 4.3, 5.5]  # x10^4 mole fraction

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(pressures_bar[:len(x_ch4_cpa)], x_ch4_cpa, color=BLUE, linewidth=1.5, label="CPA")
ax.plot(pressures_bar[:len(x_ch4_srk)], x_ch4_srk, color=ORANGE, linewidth=1.5, linestyle="--", label="SRK")
ax.scatter(exp_P, exp_x, color="black", s=25, zorder=5, label="Exp.")
ax.set_xlabel("Pressure (bar)")
ax.set_ylabel(r"$x_{\rm CH_4}$ in water ($\times 10^4$)")
ax.set_title("Methane solubility in water at 25\u00b0C")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch09_02_methane_solubility_water.png")

Saved: fig_ch09_02_methane_solubility_water.png


## Figure 9.3 — Water Content of Natural Gas vs Pressure (CPA)

In [6]:
P_range = np.arange(20, 320, 20)
temps_to_plot = [25, 50, 80]
colors_list = [BLUE, ORANGE, GREEN]

fig, ax = plt.subplots(figsize=(3.5, 3.0))

for T_C, col in zip(temps_to_plot, colors_list):
    yw_list = []
    for P in P_range:
        try:
            f = SystemSrkCPAstatoil(273.15 + float(T_C), float(P))
            f.addComponent("methane", 0.80)
            f.addComponent("ethane", 0.07)
            f.addComponent("propane", 0.03)
            f.addComponent("CO2", 0.02)
            f.addComponent("water", 0.08)
            f.setMixingRule(10)
            f.setMultiPhaseCheck(True)
            ops = ThermodynamicOperations(f)
            ops.TPflash()
            f.initProperties()
            y_w = float(f.getPhase("gas").getComponent("water").getx()) * 1e6
            yw_list.append(y_w)
        except Exception:
            yw_list.append(np.nan)
    ax.semilogy(P_range, yw_list, color=col, linewidth=1.5, label=f"{T_C}\u00b0C")

ax.set_xlabel("Pressure (bar)")
ax.set_ylabel("Water in gas (ppm mole)")
ax.set_title("Water content of natural gas (CPA)")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3, which="both")
fig.tight_layout()
save(fig, "fig_ch09_03_natgas_water_content.png")

Saved: fig_ch09_03_natgas_water_content.png
